# 🔗 05. Unificación de Datos (Merge / Data Blending)

## 🎯 Objetivo de este Notebook
Crear una **Tabla Maestra (Capa Gold)** uniendo la información de las interacciones de marketing (Banco) con la información demográfica (Clientes).

**Archivos de Entrada (Capa Silver):**
1. `bank_clean.csv` (Datos de marketing telefónico).
2. `customer_clean.csv` (Datos socioeconómicos de clientes).

**🔑 Clave de Unión:** Columna `ID`.
**🚀 Estrategia:** Realizaremos un cruce de intersección (*Inner Merge*). Esto asegura que nuestro dataset analítico final solo contenga registros que posean información completa en ambos lados, evitando la propagación de nulos masivos en el análisis.


In [1]:
# -------------------------------------------------------------------------
# 1. CARGA DE LIBRERÍAS Y DATOS LIMPIOS
# -------------------------------------------------------------------------
import pandas as pd

# Configuración para ver todas las columnas sin puntos suspensivos
pd.set_option('display.max_columns', None)

# Rutas de lectura
ruta_banco = '../data/procesados/bank_clean.csv'
ruta_clientes = '../data/procesados/customer_clean.csv'

# Cargar DataFrames
df_banco = pd.read_csv(ruta_banco)
df_clientes = pd.read_csv(ruta_clientes)

print("✅ Datos cargados correctamente.")
print(f"📊 Filas tabla Banco:    {df_banco.shape[0]}")
print(f"📊 Filas tabla Clientes: {df_clientes.shape[0]}\n")

# Vista previa para comprobar la existencia de la clave primaria 'ID'
display(df_banco.head(2))
display(df_clientes.head(2))


✅ Datos cargados correctamente.
📊 Filas tabla Banco:    43000
📊 Filas tabla Clientes: 20115



,age,job,marital,education,default,housing,loan,contact,duration,campaign,pdays,previous,poutcome,emp_var_rate,cons_price_idx,cons_conf_idx,euribor3m,nr_employed,y,date,latitude,longitude,ID
0,NaN,housemaid,married,basic_4y,0.0,0.0,0.0,telephone,261,1,999,0,nonexistent,1.1,93.994,NaN,4.857,5191.0,no,2019-08-02,41.495,-71.233,089b39d8-e4d0-461b-87d4-814d71e0e079
1,57.0,services,married,high_school,NaN,0.0,0.0,telephone,149,1,999,0,nonexistent,1.1,93.994,NaN,NaN,5191.0,no,2016-09-14,34.601,-83.923,e9d37224-cb6f-4942-98d7-46672963d097


,income,kidhome,teenhome,dt_customer,numwebvisitsmonth,ID
0,161770,1,0,2012-04-04,29,089b39d8-e4d0-461b-87d4-814d71e0e079
1,85477,1,1,2012-12-30,7,e9d37224-cb6f-4942-98d7-46672963d097


## 🕵️‍♂️ Paso 1: Auditoría de Claves (Control de Huérfanos)
Antes de realizar la unión, vamos a cruzar los conjuntos de IDs para entender cuántos registros van a quedar fuera. Esto se conoce como un análisis de "IDs Huérfanos" y es una buena práctica para asegurar la calidad de nuestro cruce relacional.


In [2]:
# -------------------------------------------------------------------------
# 2. VALIDACIÓN DE CLAVES (ORPHAN CHECK)
# -------------------------------------------------------------------------
ids_banco = set(df_banco['ID'])
ids_clientes = set(df_clientes['ID'])

solo_en_banco = len(ids_banco - ids_clientes)
solo_en_clientes = len(ids_clientes - ids_banco)

print("🔍 ANÁLISIS DE COINCIDENCIAS (Clave: ID):")
print(f"   - IDs únicos en Banco: {len(ids_banco)}")
print(f"   - IDs únicos en Clientes: {len(ids_clientes)}")
print(f"   - ⚠️ IDs sin cliente asociado (Huérfanos en Banco): {solo_en_banco}")
print(f"   - ⚠️ IDs sin interacciones (Huérfanos en Clientes): {solo_en_clientes}")
print("\n💡 Conclusión: Al usar un 'inner join', se perderán los registros huérfanos, garantizando un dataset limpio y emparejado al 100%.")


🔍 ANÁLISIS DE COINCIDENCIAS (Clave: ID):
   - IDs únicos en Banco: 43000
   - IDs únicos en Clientes: 20115
   - ⚠️ IDs sin cliente asociado (Huérfanos en Banco): 22982
   - ⚠️ IDs sin interacciones (Huérfanos en Clientes): 97

💡 Conclusión: Al usar un 'inner join', se perderán los registros huérfanos, garantizando un dataset limpio y emparejado al 100%.


## ⚙️ Paso 2: Unión de Tablas (Merge)
Ejecutamos el `inner join` utilizando la clave `ID`.


In [3]:
# -------------------------------------------------------------------------
# 3. UNIÓN DE TABLAS (INNER MERGE)
# -------------------------------------------------------------------------
df_final = pd.merge(
    left=df_banco, 
    right=df_clientes, 
    on='ID', 
    how='inner'
)

print("✅ Unión realizada con éxito.")
print(f"📊 Dimensiones de la Tabla Maestra: {df_final.shape}")
print(f"   (Filas: {df_final.shape[0]}, Columnas: {df_final.shape[1]})")

# Muestra las primeras filas para verificar que las columnas demográficas y bancarias conviven
display(df_final.head(3))


✅ Unión realizada con éxito.
📊 Dimensiones de la Tabla Maestra: (20018, 28)
   (Filas: 20018, Columnas: 28)


,age,job,marital,education,default,housing,loan,contact,duration,campaign,pdays,previous,poutcome,emp_var_rate,cons_price_idx,cons_conf_idx,euribor3m,nr_employed,y,date,latitude,longitude,ID,income,kidhome,teenhome,dt_customer,numwebvisitsmonth
0,NaN,housemaid,married,basic_4y,0.0,0.0,0.0,telephone,261,1,999,0,nonexistent,1.1,93.994,NaN,4.857,5191.0,no,2019-08-02,41.495,-71.233,089b39d8-e4d0-461b-87d4-814d71e0e079,161770,1,0,2012-04-04,29
1,57.0,services,married,high_school,NaN,0.0,0.0,telephone,149,1,999,0,nonexistent,1.1,93.994,NaN,NaN,5191.0,no,2016-09-14,34.601,-83.923,e9d37224-cb6f-4942-98d7-46672963d097,85477,1,1,2012-12-30,7
2,37.0,services,married,high_school,0.0,1.0,0.0,telephone,226,1,999,0,nonexistent,1.1,93.994,NaN,4.857,5191.0,no,2019-02-15,34.939,-94.847,3f9f49b5-e410-4948-bf6e-f9244f04918b,147233,1,1,2012-02-02,5


## 💾 Paso 3: Exportación a la Capa Gold (Tabla Analítica)
Guardamos el dataset maestro bajo el nombre `datos_finales_analisis.csv`. Con este paso, damos por **concluida la fase de Data Engineering (ETL)** y dejamos los datos preparados para la fase de Análisis y Visualización (Data Science / BI).
